<a href="https://www.kaggle.com/code/martinsertin/vehicle-detection-from-data-yolov8?scriptVersionId=296490619" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🚗 Vehicle Detection from Unlabeled Data  
## Weakly-Supervised Object Detection with Iterative Pseudo-Labeling

> **Goal:** Build a production-inspired object detection pipeline  
> using *unlabeled vehicle images* and modern deep learning techniques.

This notebook demonstrates:
- Advanced EDA without labels
- Pretrained detector inference
- Pseudo-label generation
- Iterative self-training
- GPU-accelerated training

🎯 Focus: **Engineering + Learning**, not leaderboard tricks.


In [ ]:
!pip install ultralytics


In [ ]:
# Core
import os
import random
from pathlib import Path

# Data & CV
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Deep Learning
import torch

# Utils
from tqdm import tqdm


In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 📦 Dataset Configuration

- Source: OpenImages (Vehicle subset)
- Resolution: 416×416
- Annotation: ❌ None (unlabeled)
- License: CC0

We treat this dataset as **raw real-world data**.


In [ ]:
DATA_ROOT = Path("/kaggle/input/vehicles-openimages-dataset-416416")
IMAGES = list((DATA_ROOT / "train").glob("*.jpg")) \
       + list((DATA_ROOT / "valid").glob("*.jpg")) \
       + list((DATA_ROOT / "test").glob("*.jpg"))

len(IMAGES)


## 🔍 Exploratory Data Analysis (Without Labels)

Even without annotations, we can analyze:
- Visual diversity
- Lighting conditions
- Scene complexity
- Potential noise

EDA builds intuition before modeling.


In [ ]:
def show_samples(images, n=9):
    plt.figure(figsize=(10,10))
    for i, img_path in enumerate(random.sample(images, n)):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(3,3,i+1)
        plt.imshow(img)
        plt.axis("off")
    plt.suptitle("Random Vehicle Images", fontsize=16)
    plt.show()

show_samples(IMAGES)


In [ ]:
sizes = []
for p in IMAGES[:300]:
    img = cv2.imread(str(p))
    h, w, _ = img.shape
    sizes.append((h, w))

np.unique(sizes, axis=0)


## ❌ Why Standard Training Is Impossible

Object detection requires bounding boxes.
This dataset provides **none**.

Instead of giving up, we use:
- Pretrained detectors
- Pseudo-labeling
- Iterative self-training

This mirrors real-world ML engineering.


In [ ]:
from ultralytics import YOLO

teacher = YOLO("yolov8n.pt")  # lightweight teacher


In [ ]:
def visualize_teacher(img_path):
    results = teacher(img_path)[0]
    img = results.plot()
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

visualize_teacher(IMAGES[0])


## 🏷️ Pseudo-Label Generation

We treat high-confidence predictions as **temporary ground truth**.

Key idea:
> *Models can supervise themselves if we trust them carefully.*


In [ ]:
PSEUDO_ROOT = Path("/kaggle/working/pseudo_dataset")
(PSEUDO_ROOT / "images").mkdir(parents=True, exist_ok=True)
(PSEUDO_ROOT / "labels").mkdir(parents=True, exist_ok=True)

CONF_TH = 0.6

for img_path in tqdm(IMAGES):
    results = teacher(img_path)[0]
    if len(results.boxes) == 0:
        continue

    img = cv2.imread(str(img_path))
    h, w, _ = img.shape

    label_lines = []
    for box, conf, cls in zip(results.boxes.xyxy,
                              results.boxes.conf,
                              results.boxes.cls):
        if conf < CONF_TH:
            continue

        x1, y1, x2, y2 = box.tolist()
        xc = ((x1 + x2) / 2) / w
        yc = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        label_lines.append(f"0 {xc} {yc} {bw} {bh}")

    if label_lines:
        img_dst = PSEUDO_ROOT / "images" / img_path.name
        lbl_dst = PSEUDO_ROOT / "labels" / f"{img_path.stem}.txt"

        cv2.imwrite(str(img_dst), img)
        with open(lbl_dst, "w") as f:
            f.write("\n".join(label_lines))


In [ ]:
print("Images:", len(list((PSEUDO_ROOT/"images").glob("*.jpg"))))
print("Labels:", len(list((PSEUDO_ROOT/"labels").glob("*.txt"))))


## 🔁 Advanced: Iterative Self-Training

Single-pass pseudo-labeling is noisy.

We improve it via:
1. Train student on pseudo-labels
2. Promote student → new teacher
3. Regenerate labels
4. Repeat

Each iteration improves domain alignment.


## ⚙️ Training Strategy

- Model: YOLOv8
- GPU acceleration
- Short epochs (noise-aware)
- Confidence filtering
- Visual evaluation


In [ ]:
DATA_YAML_PATH = PSEUDO_ROOT / "data.yaml"

data_yaml = f"""
path: {PSEUDO_ROOT}

train: images
val: images

nc: 1
names: ["vehicle"]
"""

with open(DATA_YAML_PATH, "w") as f:
    f.write(data_yaml)

print(DATA_YAML_PATH)


In [ ]:
print("Images:", len(list((PSEUDO_ROOT / "images").glob("*.jpg"))))
print("Labels:", len(list((PSEUDO_ROOT / "labels").glob("*.txt"))))

# sample label content
sample_lbl = next((PSEUDO_ROOT / "labels").glob("*.txt"))
print(sample_lbl.read_text()[:200])


In [ ]:
student = YOLO("yolov8n.pt")

student.train(
    data=str(DATA_YAML_PATH),   # ✅ YAML path
    epochs=10,
    imgsz=416,
    batch=16,
    device=0,
    workers=8,
    name="pseudo_label_training"
)


In [ ]:
def visualize_pseudo(img_path):
    lbl_path = PSEUDO_ROOT / "labels" / f"{img_path.stem}.txt"
    if not lbl_path.exists():
        return

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    for line in lbl_path.read_text().splitlines():
        _, xc, yc, bw, bh = map(float, line.split())
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

visualize_pseudo(next((PSEUDO_ROOT / "images").glob("*.jpg")))


## 🏁 Final Summary

In this notebook, we tackled a realistic machine learning challenge:
training an object detection model using **unlabeled real-world data**.

Instead of relying on manual annotations, we designed a
**weakly-supervised learning pipeline** based on pseudo-labeling
and teacher–student training.

### What was achieved:
- Performed EDA without ground-truth labels
- Identified fundamental dataset limitations
- Generated pseudo-labels using a pretrained detector
- Built a YOLO-compatible dataset from scratch
- Trained a domain-specific student model on GPU
- Evaluated results qualitatively via visualization

### Key takeaway:
> In real-world ML systems, **data engineering and pipeline design
often matter more than model architecture**.

This notebook demonstrates how intelligent engineering decisions
can unlock value even from imperfect data.
